<a href="https://colab.research.google.com/github/anuragN2107/Crypto-Anomaly-Detector/blob/main/Real_Time_High_Frequency_Financial_Anomaly_Detector_using_LSTM_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Project Title
**CryptoPulse: Real-Time Cryptocurrency High-Frequency Anomaly Detector**


# 2. Business Problem
High-Frequency Trading (HFT) and automated liquidity providers in cryptocurrency markets lose millions of dollars annually due to "flash crashes," market manipulation (such as pump-and-dump schemes), and sudden liquidity drains. Standard technical indicators (like RSI or MACD) calculate metrics over fixed historical windows, lagging significantly behind real-time microstructural anomalies.


#3.Objective
To build a streaming-ready deep learning pipeline capable of ingesting high-frequency transactional data, modeling the complex temporal dependencies of market sequences, and identifying behavioral price/volume anomalies within milliseconds to alert automated trading infrastructure.


#4. Why RNN/LSTM?
Traditional machine learning models (like Random Forests or XGBoost) treat data points as independent. Financial market data, however, is a time series where current price action heavily depends on recent historical context.

* **Sequential Memory:** Long Short-Term Memory (LSTM) networks—a specialized type of Recurrent Neural Network (RNN)—excel here because they use "gates" to retain long-term dependencies and forget irrelevant market noise.

* **Pattern Recognition:** LSTMs can learn what a normal market regime looks like over a lookback window (e.g., the last 60 seconds of tick data). When the actual market behavior drastically diverges from the LSTM's prediction, we flag it as an anomaly.

# 5. Methodology:
* **Data Ingestion & Engineering:** Sourcing or simulating raw order-book/tick data and engineering sequence features.

* **Sequential Modeling:** Utilizing an LSTM/RNN autoencoder architecture to learn baseline market behaviors.

* **Reconstruction-Based Anomaly Detection:** Flagging instances where the actual market state deviates heavily from the network’s predicted baseline.

* **Containerization:** Packaging the inference engine, background data updater, and user dashboard into an isolated Docker network.

* **Deployment:** Hosting the container on Hugging Face Spaces with an eye-catching, high-contrast, professional-grade interface.



# 6. Tech Stack & Tools
* **Deep Learning Engine:** PyTorch (torch.nn) utilizing memory-gated $LSTM$ sequence processing.
* **Data Processing & Prep:** Pandas, NumPy, and Scikit-Learn (MinMaxScaler).
* **UI & Data Visuals:** Streamlit combined with custom CSS3 neon styling and live Plotly vector charts.
* **Deployment Network:** Isolated multi-stage Docker container optimized for Hugging Face Spaces CPU infrastructure.

#Step 1: Install Dependencies & Import Modules

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import joblib
import time
np.random.seed(42)
torch.manual_seed(42)

#Step 2: Synthetic HFT Data Generation

In [4]:
def generate_hft_data(n_points=5000):
    timestamp = pd.date_range(start="2026-01-01", periods=n_points, freq="s")

    # random walk simulation
    returns = np.random.normal(0, 0.001, n_points)
    price = 50000 * np.exp(np.cumsum(returns))
    volume = np.random.gamma(shape=2, scale=50, size=n_points) + 10

    df = pd.DataFrame({'Price': price, 'Volume': volume}, index=timestamp)

    # Flash Crash (Steps 1500-1510)
    df.iloc[1500:1510, df.columns.get_loc('Price')] *= 0.85
    df.iloc[1500:1510, df.columns.get_loc('Volume')] *= 15

    #  Wash Trading / Volume Pump (Steps 3500-3530)
    df.iloc[3500:3530, df.columns.get_loc('Volume')] *= 30

    return df

df = generate_hft_data()
print(df.head())

                            Price      Volume
2026-01-01 00:00:00  50024.841877   68.863480
2026-01-01 00:00:01  50017.925705   67.358273
2026-01-01 00:00:02  50050.332236  150.152422
2026-01-01 00:00:03  50126.618465   35.175377
2026-01-01 00:00:04  50114.882522   54.355967


#Step 3: Sequence Preprocessing

In [5]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df.values)

def create_sequences(data, window_size=30):
    sequences = []
    for i in range(len(data) - window_size):
        sequences.append(data[i:i+window_size])
    return np.array(sequences)

WINDOW_SIZE = 30
X = create_sequences(scaled_data, WINDOW_SIZE)
X_tensor = torch.tensor(X, dtype=torch.float32)
print("Sequence Tensor Shape:", X_tensor.shape)

Sequence Tensor Shape: torch.Size([4970, 30, 2])


#Step 4: Constructing the LSTM Autoencoder Network

In [6]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, no_features, embedding_dim=16):
        super(LSTMAutoencoder, self).__init__()
        self.seq_len = seq_len
        self.no_features = no_features

        # Encoder
        self.encoder_lstm = nn.LSTM(input_size=no_features, hidden_size=embedding_dim, num_layers=1, batch_first=True)
        # Decoder
        self.decoder_lstm = nn.LSTM(input_size=embedding_dim, hidden_size=embedding_dim, num_layers=1, batch_first=True)
        self.output_linear = nn.Linear(embedding_dim, no_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder_lstm(x)

        repeat_hidden = hidden.permute(1, 0, 2).repeat(1, self.seq_len, 1)

        x_decoded, _ = self.decoder_lstm(repeat_hidden)
        output = self.output_linear(x_decoded)
        return output

model = LSTMAutoencoder(seq_len=WINDOW_SIZE, no_features=2)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

#Step 5: Training Loop & Saving Artifacts

In [7]:
epochs = 10
batch_size = 64

for epoch in range(epochs):
    model.train()
    permutation = torch.randperm(X_tensor.size()[0])
    epoch_loss = 0

    for i in range(0, X_tensor.size()[0], batch_size):
        optimizer.zero_grad()
        indices = permutation[i:i+batch_size]
        batch_x = X_tensor[indices]

        predictions = model(batch_x)
        loss = criterion(predictions, batch_x)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss / X_tensor.size(0):.6f}")

torch.save(model.state_dict(), 'lstm_anomaly_model.pth')
joblib.dump(scaler, 'data_scaler.pkl')
print("Artifacts successfully serialized!")

Epoch 1/10 | Loss: 0.053766
Epoch 2/10 | Loss: 0.007621
Epoch 3/10 | Loss: 0.002160
Epoch 4/10 | Loss: 0.001467
Epoch 5/10 | Loss: 0.001303
Epoch 6/10 | Loss: 0.001280
Epoch 7/10 | Loss: 0.001210
Epoch 8/10 | Loss: 0.001171
Epoch 9/10 | Loss: 0.001126
Epoch 10/10 | Loss: 0.001041
Artifacts successfully serialized!
